# 06 · Improved pipeline (v2): target ≥ 90 % accuracy

Run this on a **GPU** (Kaggle notebook with *GPU T4 x2* or *P100*, or Google Colab with a T4/L4/A100).
It does not run on a CPU in reasonable time.

| Step | What | T4 time (approx.) |
|---|---|---|
| 1 | Install `timm`, `segmentation-models-pytorch` | 1 min |
| 2 | Data: HAM10000 (+ ISIC 2018 Task 1 masks, + ISIC 2019 extra images) | 20-40 min download |
| 3 | Train U-Net lesion segmentation | ~40 min |
| 4 | Crop to lesion + Shades-of-Gray colour constancy, cache 512 px | ~20 min |
| 5 | Train 3 classifiers at 384 px (EfficientNetV2-S, ConvNeXt-S, EfficientNet-B4-NS) | 1.5-3 h each |
| 6 | Ensemble + calibration on validation, evaluate once on test | 1 min |

The HAM10000 **test split is the same one used for the v1 results** (82.0 %), so the numbers are comparable.

**Budget options:** set `USE_ISIC2019 = False` to train on HAM10000 only (≈3x faster, typically 1-3 points lower);
set `USE_UNET = False` to use the Otsu crop instead of the U-Net (skips the 10 GB ISIC 2018 download).

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
PROJECT = "/kaggle/working/skin-cancer-detection" if ON_KAGGLE else "/content/skin-cancer-detection"
BIG = "/tmp/isic" if ON_KAGGLE else "/content/isic"       # large downloads go here (not saved to output)

USE_UNET = True        # ISIC 2018 Task 1 masks -> U-Net lesion crop (else Otsu crop)
USE_ISIC2019 = True    # ~16k extra training images from ISIC 2019 (leakage-filtered)
ARCHS = ["tf_efficientnetv2_s.in21k_ft_in1k",
         "convnext_small.fb_in22k_ft_in1k_384",
         "tf_efficientnet_b4.ns_jft_in1k"]
EPOCHS = 25
WORKERS = 4 if ON_KAGGLE else 2
os.makedirs(BIG, exist_ok=True)

## 1 · Project code and libraries
Upload the project zip (Kaggle: *Add data → Upload*; Colab: the file panel) and unzip it to `PROJECT`,
or `git clone` your repository there.

In [ ]:
# Example for an uploaded zip - adjust the path to where your zip landed:
# !unzip -q -o /kaggle/input/<your-upload>/skin-cancer-detection.zip -d /kaggle/working/
# !unzip -q -o /content/skin-cancer-detection.zip -d /content/
!pip -q install timm segmentation-models-pytorch
%cd {PROJECT}
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!")

## 2 · Data
**HAM10000** - on Kaggle, *Add data* → search `skin-cancer-mnist-ham10000` (kmader). On Colab, use the
project's `scripts/download_data.py` (needs your Kaggle token).

**ISIC 2018 Task 1** (segmentation masks) and **ISIC 2019** (extra training images) - official ISIC challenge
downloads (https://challenge.isic-archive.com/data/). If you prefer, add Kaggle mirrors of them via *Add data*
and point `TASK1_IMG`, `TASK1_MASK`, `ISIC2019_DIR` at those folders instead of downloading.

In [ ]:
# HAM10000 -> data/raw
ham_in = "/kaggle/input/skin-cancer-mnist-ham10000"
if ON_KAGGLE and os.path.exists(ham_in):
    !rm -rf data/raw && ln -s {ham_in} data/raw
else:
    !python scripts/download_data.py
!ls data/raw | head

In [ ]:
S3 = "https://isic-challenge-data.s3.amazonaws.com"
def fetch(url, dest):
    if os.path.exists(dest) and os.listdir(dest):
        print("exists:", dest); return
    os.makedirs(dest, exist_ok=True)
    z = os.path.join(BIG, os.path.basename(url))
    subprocess.run(["wget", "-q", "-c", "-O", z, url], check=True)
    if z.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", dest], check=True); os.remove(z)
    else:
        os.replace(z, os.path.join(dest, os.path.basename(url)))

TASK1_IMG, TASK1_MASK = f"{BIG}/task1_img", f"{BIG}/task1_mask"
ISIC2019_DIR = f"{BIG}/isic2019"
if USE_UNET:
    fetch(f"{S3}/2018/ISIC2018_Task1_Training_GroundTruth.zip", TASK1_MASK)
    fetch(f"{S3}/2018/ISIC2018_Task1-2_Training_Input.zip", TASK1_IMG)        # 10.4 GB
if USE_ISIC2019:
    fetch(f"{S3}/2019/ISIC_2019_Training_GroundTruth.csv", ISIC2019_DIR)
    fetch(f"{S3}/2019/ISIC_2019_Training_Metadata.csv", ISIC2019_DIR)
    fetch(f"{S3}/2019/ISIC_2019_Training_Input.zip", ISIC2019_DIR)             # 9.1 GB
!df -h /tmp /kaggle/working 2>/dev/null | tail -2

## 3 · Lesion segmentation (U-Net, EfficientNet-B4 encoder)

In [ ]:
if USE_UNET:
    !python improved/segmentation.py train --task1-images {TASK1_IMG} --task1-masks {TASK1_MASK} --epochs 25 --workers {WORKERS}
    !rm -rf {TASK1_IMG}          # free 10 GB once the U-Net is trained

## 4 · Crop to lesion + colour constancy → `data/v2/` cache

In [ ]:
isic_arg = f"--isic2019 {ISIC2019_DIR}" if USE_ISIC2019 else ""
unet_arg = "" if USE_UNET else "--no-unet"
!python improved/segmentation.py cache {isic_arg} {unet_arg}

In [ ]:
# Visual check of the crops
import matplotlib.pyplot as plt, glob, cv2, random
fs = random.sample(glob.glob("data/v2/*.jpg"), 8)
fig, ax = plt.subplots(1, 8, figsize=(20, 3))
for a, f in zip(ax, fs):
    a.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); a.set_title(os.path.basename(f)[:12], fontsize=8); a.axis("off")

## 5 · Train the classifiers
Each run saves `models/v2/<tag>.pth` and TTA probabilities for val/test in `results/v2/<tag>/`.
Kaggle sessions stop after 12 h: if needed, run one architecture per session (outputs persist in `/kaggle/working`).

In [ ]:
for arch in ARCHS:
    !python improved/train_cls.py --arch {arch} --meta {isic_arg} --epochs {EPOCHS} --workers {WORKERS}

## 6 · Ensemble + calibration (fitted on validation only) → final test metrics

In [ ]:
!python improved/ensemble.py --search-subsets
import json; m = json.load(open("results/v2/ensemble/ensemble_metrics.json"))
import pandas as pd
pd.DataFrame({k: v for k, v in m.items() if k.startswith("test_")}).T[
    ["accuracy", "balanced_accuracy", "macro_f1", "macro_roc_auc", "bin_accuracy", "bin_sensitivity", "bin_specificity"]].round(4)

In [ ]:
print(open("results/v2/ensemble/classification_report.txt").read())

## 7 · Explainability check with the v2 predictor (Grad-CAM)

In [ ]:
sys.path.insert(0, "improved")
from predict_v2 import EnsemblePredictor
from preprocess import load_image_rgb
P = EnsemblePredictor()
fs = sorted(glob.glob("sample_images/*.jpg"))
fig, ax = plt.subplots(2, len(fs), figsize=(3 * len(fs), 6))
for i, f in enumerate(fs):
    r = P.predict_array(load_image_rgb(f))
    ax[0, i].imshow(r["preprocessed"]); ax[0, i].set_title(os.path.basename(f)[:10], fontsize=8)
    ax[1, i].imshow(r["overlay"]); ax[1, i].set_title(f"{r['pred_class']} {r['confidence']:.2f}", fontsize=8)
    ax[0, i].axis("off"); ax[1, i].axis("off")

## 8 · Use it in the app
In `app.py`, replace `from predict import Predictor` with
`from improved.predict_v2 import EnsemblePredictor` and construct `EnsemblePredictor()` in `get_predictor`.
Copy `models/v2/` and `results/v2/ensemble/` back to your machine.